In [ ]:
%matplotlib inline
import os
import cv2
from skimage.morphology import remove_small_objects
import matplotlib.pyplot as plt
import numpy as np
from tifffile import tifffile
import ipywidgets as widgets
from ipywidgets import interact

In [ ]:
def process_and_display(gain, threshold, kernel_size):
    infile = "014.tif"
    image_path = f"/net/birdstore/Active_Atlas_Data/data_root/pipeline_data/DK283/preps/C2/polygons"
    image_inpath = os.path.join(image_path, infile)
    with tifffile.TiffFile(image_inpath) as tif:
        img = tif.asarray()

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    #left side image, original * gain so you can see it
    modified_img = img.astype(np.uint32) * gain
    gained_img = np.clip(modified_img, 0, 65535).astype(np.uint16)

    axes[0].imshow(gained_img, cmap="gray")
    axes[0].set_title(f"Original with gain: {gain}")
    axes[0].axis("off")

    #right side image with threshold and gain
    #clipped = np.clip(img, threshold, 65535)
    #modified_values = clipped.astype(np.uint32) * gain
    #adjusted = np.clip(modified_values, 0, 65535).astype(np.uint16)
    binary_mask = img > threshold
    # Apply morphological opening
    if kernel_size > 0:
        binary_mask = remove_small_objects(binary_mask, max_size=kernel_size)
    pixel_count = np.count_nonzero(binary_mask)
    int8_arr = (binary_mask * 255).astype(np.int8)

    
    axes[1].imshow(int8_arr, cmap="gray_r")
    axes[1].set_title(f"Threshold: {threshold} with pixel count: {pixel_count}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
interact(process_and_display,
    gain=widgets.IntSlider(min=0, max=15, step=1, value=10, description="Gain"),
    threshold=widgets.IntSlider(min=0, max=15000, step=10, value=0, description="Threshold"),
    kernel_size=widgets.IntSlider(min=0, max=500, step=5, value=0, description="Removal size")
)
